# UK Biobank Rare Variant Burden Analysis Pipeline

Complete workflow for gene-based rare variant burden testing using REGENIE.

**Pipeline Steps:**
1. Load UK Biobank dataset
2. Define cases and controls
3. Prepare phenotype files
4. Extract and filter genotypes by MAF
5. Create gene annotations
6. Run REGENIE Step 1 (null model)
7. Run REGENIE Step 2 (burden tests)

**Author:** Marzieh Khani 

**Date:** 03/09/2026


## Configuration

In [ ]:
import pyspark
import dxdata
import dxpy
import pandas as pd
import numpy as np
import subprocess
import os

sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

# Configuration
ANCESTRIES = ["EUR", "AFR", "EAS", "SAS", "AAC", "AJ", "CAS", "CAH", "MDE", "AMR", "FIN"]
PRIORITY = "EUR"
MAF_THRESHOLDS = {"001": 0.01, "003": 0.03, "005": 0.05}
VARIANT_TYPES = ["all", "coding"]


## Load Dataset and Define Cases

In [ ]:
# Find and load dataset
dispensed_dataset_id = dxpy.find_one_data_object(
    typename="Dataset",
    name="app*.dataset",
    folder="/",
    name_mode="glob"
)["id"]

dataset = dxdata.load_dataset(id=dispensed_dataset_id)
participant = dataset["participant"]

print(f"Dataset loaded: {dispensed_dataset_id}")

# Define fields to extract
field_names = [
    "eid",           # Participant ID
    "p31",           # Genetic sex
    "p34",           # Birth year
    "p21022",        # Age at recruitment
    "p42020",        # Date of AD diagnosis
    "p22009_a1",     # PC1
    "p22009_a2",     # PC2
    "p22009_a3",     # PC3
    "p22009_a4",     # PC4
    "p22009_a5",     # PC5
    "p22009_a6",     # PC6
    "p22009_a7",     # PC7
    "p22009_a8",     # PC8
    "p22009_a9",     # PC9
    "p22009_a10",    # PC10
    "p40000_i0",     # Date of death
]

print("Retrieving participant data...")
df_cases = participant.retrieve_fields(
    names=field_names,
    coding_values="replace",
    engine=dxdata.connect()
)
df_cases = df_cases.toPandas()

# Rename columns
df_cases.rename(columns={
    "eid": "ID",
    "p31": "GENETIC_SEX",
    "p34": "BIRTH_YEAR",
    "p21022": "AGE_OF_RECRUIT",
    "p42020": "AD_DATE",
    "p22009_a1": 'PC1',
    "p22009_a2": 'PC2',
    "p22009_a3": 'PC3',
    "p22009_a4": 'PC4',
    "p22009_a5": 'PC5',
    "p22009_a6": 'PC6',
    "p22009_a7": 'PC7',
    "p22009_a8": 'PC8',
    "p22009_a9": 'PC9',
    "p22009_a10": 'PC10',
    "p40000_i0": "DATE_OF_DEATH",
}, inplace=True)

df_cases["ID"] = pd.to_numeric(df_cases["ID"])

# Extract AD cases (participants with AD diagnosis date)
df_ad = df_cases[~df_cases['AD_DATE'].isna()].copy()
df_ad = df_ad[[
    "ID", "GENETIC_SEX", "BIRTH_YEAR", "AGE_OF_RECRUIT",
    "AD_DATE", "PC1", "PC2", "PC3", "PC4", "PC5",
    "PC6", "PC7", "PC8", "PC9", "PC10", "DATE_OF_DEATH"
]]

print(f"Total participants retrieved: {len(df_cases)}")
print(f"AD cases identified: {len(df_ad)}")
print(f"  Mean age at recruitment: {df_ad['AGE_OF_RECRUIT'].mean():.1f}")
print(f"  Sex distribution:")
print(df_ad['GENETIC_SEX'].value_counts())



## Select Controls

In [ ]:
# Define neurological exclusion fields
neurological_fields = [
    'p131012', 'p131016', 'p131018', 'p131020', 'p131022', 'p131024',
    'p131026', 'p131028', 'p131030', 'p131036', 'p131038', 'p131040',
    'p131042', 'p131046', 'p131056', 'p131058', 'p131062', 'p131066',
    'p131068', 'p131070', 'p131074', 'p131076', 'p131078', 'p131080',
    'p131082', 'p131084', 'p131086', 'p131088', 'p131090', 'p131092',
    'p131094', 'p131096', 'p131098', 'p131100', 'p131102', 'p131104',
    'p131106', 'p131108', 'p131110', 'p131112', 'p131114', 'p131116',
    'p131120', 'p131122', 'p131124', 'p131126',
    'p42018', 'p42020', 'p42022', 'p42024', 'p42028',
    'p42030', 'p42032', 'p42034', 'p42036'
]

# Parent illness fields
parent_fields = [
    'p20110_i0', 'p20110_i1', 'p20110_i2', 'p20110_i3',
    'p20107_i0', 'p20107_i1', 'p20107_i2', 'p20107_i3'
]

# All fields needed for controls
control_field_names = (
    ['eid', 'p22006', 'p21022', 'p31', 'p34', 'p40000_i0'] +
    ['p22009_a' + str(i) for i in range(1, 11)] +
    neurological_fields +
    parent_fields
)

print("Retrieving potential control participants...")
df_control = participant.retrieve_fields(
    names=control_field_names,
    coding_values="replace",
    engine=dxdata.connect(dialect="hive+pyspark")
)
df_control = df_control.toPandas()

print(f"Initial participants: {len(df_control)}")

# Filter 1: Remove participants with neurological conditions
print("Filtering: Removing participants with neurological conditions...")
for field in neurological_fields:
    if field in df_control.columns:
        df_control = df_control[df_control[field].isnull()]

print(f"  After neurological filter: {len(df_control)}")

# Filter 2: Remove participants with family history of AD or PD
print("Filtering: Removing participants with family history of AD/PD...")
for col in parent_fields:
    if col in df_control.columns:
        df_control[col] = df_control[col].apply(
            lambda x: x if isinstance(x, list) else []
        )

condition = lambda row: all(
    "Alzheimer's disease/dementia" not in illnesses and
    "Parkinson's disease" not in illnesses
    for col in parent_fields
    for illnesses in [row.get(col, [])]
)

df_control = df_control[df_control.apply(condition, axis=1)]
print(f"  After family history filter: {len(df_control)}")

# Filter 3: Age filter (≥65 at recruitment)
print("Filtering: Age ≥65 at recruitment...")
df_control = df_control[df_control['p21022'] >= 65]
print(f"  After age filter: {len(df_control)}")

# Rename and select columns
df_control = df_control[[
    'eid', 'p21022', 'p31',
    'p22009_a1', 'p22009_a2', 'p22009_a3', 'p22009_a4', 'p22009_a5',
    'p22009_a6', 'p22009_a7', 'p22009_a8', 'p22009_a9', 'p22009_a10',
    'p34', 'p22006', 'p40000_i0'
]]

df_control.rename(columns={
    'eid': 'ID',
    'p21022': 'AGE_OF_RECRUIT',
    'p31': 'GENETIC_SEX',
    'p22009_a1': 'PC1', 'p22009_a2': 'PC2', 'p22009_a3': 'PC3',
    'p22009_a4': 'PC4', 'p22009_a5': 'PC5', 'p22009_a6': 'PC6',
    'p22009_a7': 'PC7', 'p22009_a8': 'PC8', 'p22009_a9': 'PC9',
    'p22009_a10': 'PC10',
    'p34': 'BIRTH_YEAR',
    'p22006': 'ETHNICITY',
    'p40000_i0': 'DATE_OF_DEATH',
}, inplace=True)

df_control["ID"] = pd.to_numeric(df_control["ID"])

print(f"Final control participants: {len(df_control)}")


## Assigning ancestry labels

In [ ]:
! dx download data/ukbb_imputed_genotypes_umap_linearsvc_predicted_labels.txt --overwrite

ancestries = pd.read_csv(
    "ukbb_imputed_genotypes_umap_linearsvc_predicted_labels.txt",
    sep="\t"
)

print(f"Loaded ancestry labels for {len(ancestries)} participants")
print("\nAncestry distribution in full dataset:")
print(ancestries['label'].value_counts())

# Add ancestry labels to cases and controls
df_control = df_control.merge(
    ancestries[["IID", "label"]],
    left_on="ID",
    right_on="IID"
).drop("IID", axis=1)

df_ad = df_ad.merge(
    ancestries[["IID", "label"]],
    left_on="ID",
    right_on="IID"
).drop("IID", axis=1)

print(f"\n Ancestry labels assigned")
print(f"\nAD cases by ancestry:")
print(df_ad['label'].value_counts())
print(f"\nControls by ancestry:")
print(df_control['label'].value_counts())

## Filtering for WGS data availability

In [ ]:
# Get ID lists from current dataframes
ids_ad = df_ad["ID"].tolist()
ids_control = df_control["ID"].tolist()

print(f"IDs before WGS filter:")
print(f"  AD cases: {len(ids_ad)}")
print(f"  Controls: {len(ids_control)}")
print(f"  Total: {len(ids_ad) + len(ids_control)}")

print("\nExtracting list of participants with WGS data...")

# Submit job to get WGS sample IDs
wgs_job = """
dx run swiss-army-knife \
  -iin="/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c1_b0_v1.psam" \
  -icmd='awk "{print \\$2}" *.psam | sort -u > plink_sample_ids.txt' \
  --instance-type mem2_ssd1_v2_x4 \
  --destination /Results/burden_temp/ \
  --name get_wgs_ids \
  --yes --brief
"""

result = subprocess.run(wgs_job, shell=True, capture_output=True, text=True)
wgs_job_id = result.stdout.strip()
print(f"Job submitted: {wgs_job_id}")

# Wait for job to complete
print("Waiting for WGS ID extraction to complete...")
! dx wait {wgs_job_id}

# Download results
print("Downloading WGS sample IDs...")
! dx download /Results/burden_temp/plink_sample_ids.txt --overwrite

# Save our cohort IDs to file
with open('cohort_ids.txt', 'w') as f:
    for iid in ids_ad + ids_control:
        f.write(f"{iid}\n")

# Filter for WGS availability
print("Filtering cohort IDs for WGS availability...")
! grep -Fwf plink_sample_ids.txt cohort_ids.txt > ids_with_wgs.txt

# Read filtered IDs
with open('ids_with_wgs.txt', 'r') as f:
    ids_with_wgs = set(int(line.strip()) for line in f)

# Update dataframes
df_ad = df_ad[df_ad["ID"].isin(ids_with_wgs)]
df_control = df_control[df_control["ID"].isin(ids_with_wgs)]

# Update ID lists
ids_ad = df_ad["ID"].tolist()
ids_control = df_control["ID"].tolist()

print(f"\n Final cohort with WGS data:")
print(f"  AD cases: {len(ids_ad)}")
print(f"  Controls: {len(ids_control)}")
print(f"  Total: {len(ids_ad) + len(ids_control)}")


## Creating phenotype files

In [ ]:
# Add phenotype column (2=case, 1=control)
df_ad['PHENO'] = 2
df_control['PHENO'] = 1

# Combine
df_total = pd.concat([df_ad, df_control], ignore_index=True)

print(f"Combined dataset:")
print(f"  Total participants: {len(df_total)}")
print(f"  Cases (PHENO=2): {(df_total['PHENO']==2).sum()}")
print(f"  Controls (PHENO=1): {(df_total['PHENO']==1).sum()}")

# Save combined file
df_total.to_csv("AD_Control_all_ancestries.csv", index=False)
! dx upload AD_Control_all_ancestries.csv --path /Results/burden_pheno/

print("\n Saved combined phenotype file")

# Create per-ancestry phenotype files (REGENIE format: FID IID PHENO)
print("\nCreating per-ancestry phenotype files...")
for anc in ANCESTRIES:
    df_anc = df_total[df_total["label"] == anc].copy()

    if len(df_anc) == 0:
        print(f"   {anc}: No participants - skipping")
        continue

    # Create FID, IID, PHENO dataframe
    df_pheno = df_anc[["ID", "ID", "PHENO"]].copy()
    df_pheno.columns = ["FID", "IID", "PHENO"]

    # Save
    pheno_file = f"PHENO_{anc}.txt"
    df_pheno.to_csv(pheno_file, sep="\t", index=False)
    ! dx upload {pheno_file} --path /Results/burden_pheno/

    n_cases = (df_anc['PHENO'] == 2).sum()
    n_controls = (df_anc['PHENO'] == 1).sum()
    print(f"   {anc}: {len(df_anc)} total ({n_cases} cases, {n_controls} controls)")

print("\n All phenotype files created and uploaded")


## Creating covariate files

In [ ]:
# Calculate AGE
# For cases: age at AD diagnosis = AD_DATE.year - BIRTH_YEAR
# For controls: current age = 2025 - BIRTH_YEAR

df_total["AD_DATE"] = pd.to_datetime(df_total["AD_DATE"], errors="coerce")
df_total["BIRTH_YEAR"] = pd.to_numeric(df_total["BIRTH_YEAR"], errors="coerce")

df_total["AGE"] = df_total.apply(
    lambda row: row["AD_DATE"].year - row["BIRTH_YEAR"]
    if row["PHENO"] == 2 and pd.notnull(row["AD_DATE"]) and pd.notnull(row["BIRTH_YEAR"])
    else 2025 - row["BIRTH_YEAR"]
    if row["PHENO"] == 1 and pd.notnull(row["BIRTH_YEAR"])
    else None,
    axis=1
)

print(f"Age calculated for {df_total['AGE'].notna().sum()}/{len(df_total)} participants")

# Create covariate file (REGENIE format: FID IID SEX AGE PC1-PC10)
df_covar = df_total[[
    "ID", "ID", "GENETIC_SEX",
    "PC1", "PC2", "PC3", "PC4", "PC5",
    "PC6", "PC7", "PC8", "PC9", "PC10",
    "AGE"
]].copy()

df_covar.columns = [
    "FID", "IID", "SEX",
    "PC1", "PC2", "PC3", "PC4", "PC5",
    "PC6", "PC7", "PC8", "PC9", "PC10",
    "AGE"
]

# Convert sex to numeric (Male=1, Female=2)
df_covar["SEX"] = df_covar["SEX"].map({"Male": 1, "Female": 2})

# Save
df_covar.to_csv("covariates.tsv", sep="\t", index=False, na_rep="NA")
! dx upload covariates.tsv --path /Results/burden_pheno/

print(" Covariate file created and uploaded")
print(f"  Columns: FID, IID, SEX, PC1-PC10, AGE")
print(f"  Participants: {len(df_covar)}")
print(f"  Missing AGE: {df_covar['AGE'].isna().sum()}")

## Creating sample ID files

In [ ]:
for anc in ANCESTRIES:
    df_anc = df_total[df_total["label"] == anc]

    if len(df_anc) == 0:
        continue

    # FID IID format (space-separated, no header)
    df_ids = df_anc[["ID", "ID"]].copy()
    df_ids.columns = ["FID", "IID"]

    id_file = f"sample_ids_{anc}.txt"
    df_ids.to_csv(id_file, sep=" ", index=False, header=False)
    ! dx upload {id_file} --path /Results/burden_pheno/

    print(f"   {anc}: {len(df_ids)} samples")

print("\n All sample ID files created and uploaded")



## Extracting WGS data for {PRIORITY}

In [ ]:
%%bash
dx mkdir -p /Results/burden_geno/EUR

for c in {1..22}; do
    dx mkdir -p /Results/burden_geno/EUR/chr${c}
done

In [ ]:
extraction_jobs = []

for chr_num in range(1, 23):
    job_cmd = f"""
dx run swiss-army-knife \\
  -iin="/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c{chr_num}_b0_v1.pgen" \\
  -iin="/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c{chr_num}_b0_v1.pvar" \\
  -iin="/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c{chr_num}_b0_v1.psam" \\
  -iin="/Results/burden_pheno/sample_ids_{PRIORITY}.txt" \\
  -icmd="plink2 --pfile ukb24308_c{chr_num}_b0_v1 \\
                --keep sample_ids_{PRIORITY}.txt \\
                --no-pheno \\
                --make-pgen \\
                --out {PRIORITY}_chr{chr_num}_raw" \\
  --instance-type mem2_ssd1_v2_x96 \\
  --destination /Results/burden_geno/{PRIORITY}/chr{chr_num}/ \\
  --name extract_{PRIORITY}_chr{chr_num} \\
  --yes --brief
"""

    result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
    job_id = result.stdout.strip()
    extraction_jobs.append(job_id)
    print(f"  Chr{chr_num}: {job_id}")

print(f"\n Submitted {len(extraction_jobs)} extraction jobs for {PRIORITY}")
print(f"\nMonitor progress: dx watch {extraction_jobs[0]}")


## QC and MAF stratification for {PRIORITY}

In [ ]:
%%bash
for c in {1..22}; do
    dx mkdir -p /Results/burden_geno/EUR/chr${c}_QC
done

In [ ]:
qc_script_template = """set -euo pipefail

PFX={anc}_chr{chr}_raw

echo "QC and MAF stratification for {anc} chr{chr}..."

# Step 1: Find multi-allelic sites (duplicate positions)
echo "Finding multi-allelic sites..."
awk 'NR>1 {{key=$1":"$2; c[key]++}} END{{for(k in c) if(c[k]>1) print k}}' ${{PFX}}.pvar > dup.pos

# Step 2: Keep only biallelic SNPs (not in dup list)
echo "Filtering to biallelic SNPs..."
awk 'BEGIN{{FS=OFS="\\t"}}
     NR==FNR {{dup[$1]=1; next}}
     NR==1 {{next}}
     {{
       key=$1":"$2
       is_snp=(length($4)==1 && length($5)==1 && $4~/^[ACGT]$/ && $5~/^[ACGT]$/)
       if(is_snp && !(key in dup)) print $3
     }}' dup.pos ${{PFX}}.pvar > keep.ids

# Step 3: Apply QC filters
echo "Applying QC filters..."
plink2 --pfile ${{PFX}} \\
       --extract keep.ids \\
       --geno 0.05 \\
       --hwe 1e-6 midp \\
       --make-pgen \\
       --out {anc}_chr{chr}_clean

# Step 4: Create three MAF datasets with MAC filter
echo "Creating MAF-stratified datasets with MAC ≥ 1..."

plink2 --pfile {anc}_chr{chr}_clean \\
       --max-maf 0.01 \\
       --mac 1 \\
       --make-pgen \\
       --out {anc}_chr{chr}_MAF001

plink2 --pfile {anc}_chr{chr}_clean \\
       --max-maf 0.03 \\
       --mac 1 \\
       --make-pgen \\
       --out {anc}_chr{chr}_MAF003

plink2 --pfile {anc}_chr{chr}_clean \\
       --max-maf 0.05 \\
       --mac 1 \\
       --make-pgen \\
       --out {anc}_chr{chr}_MAF005

echo ""
echo " Chr{chr} complete!"
echo "Variant counts:"
echo "  Clean: $(tail -n+2 {anc}_chr{chr}_clean.pvar | wc -l)"
echo "  MAF≤0.01 (MAC≥1): $(tail -n+2 {anc}_chr{chr}_MAF001.pvar | wc -l)"
echo "  MAF≤0.03 (MAC≥1): $(tail -n+2 {anc}_chr{chr}_MAF003.pvar | wc -l)"
echo "  MAF≤0.05 (MAC≥1): $(tail -n+2 {anc}_chr{chr}_MAF005.pvar | wc -l)"
"""

qc_jobs = []

for chr_num in range(1, 23):
    # Create script
    script_content = qc_script_template.format(anc=PRIORITY, chr=chr_num)
    script_name = f"qc_{PRIORITY}_chr{chr_num}.sh"

    with open(script_name, "w") as f:
        f.write(script_content)

    # Submit job 
    job_cmd = f"""
dx run swiss-army-knife \\
  -iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}/{PRIORITY}_chr{chr_num}_raw.pgen" \\
  -iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}/{PRIORITY}_chr{chr_num}_raw.pvar" \\
  -iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}/{PRIORITY}_chr{chr_num}_raw.psam" \\
  -icmd="$(cat {script_name})" \\
  --instance-type mem2_ssd1_v2_x96 \\
  --destination /Results/burden_geno/{PRIORITY}/chr{chr_num}_QC/ \\
  --name QC_{PRIORITY}_chr{chr_num} \\
  --yes --brief
"""

    result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
    job_id = result.stdout.strip()
    qc_jobs.append(job_id)
    print(f"  Chr{chr_num}: {job_id}")

print(f"\n Submitted {len(qc_jobs)} QC+MAF jobs for {PRIORITY}")
print(f" These will start automatically after extraction completes")


## Merging chromosomes by MAF threshold for {PRIORITY}

In [ ]:
%%bash
dx mkdir -p /Results/burden_geno/EUR/merged

In [ ]:
merge_jobs = {}

for maf_code in MAF_THRESHOLDS.keys():
    print(f"\nPreparing merge for MAF {MAF_THRESHOLDS[maf_code]}...")
    
    # Create merge list file (chromosomes 2-22)
    merge_list_file = f"merge_{PRIORITY}_MAF{maf_code}.txt"
    with open(merge_list_file, 'w') as f:
        for chr_num in range(2, 23):
            f.write(f"{PRIORITY}_chr{chr_num}_MAF{maf_code}\n")
    
    # Upload merge list
    subprocess.run(f"dx upload {merge_list_file} --path /Results/burden_geno/{PRIORITY}/", 
                   shell=True, check=True)
    
    # Create merge script 
    merge_script = f"""#!/bin/bash
set -euo pipefail

# List all pgen files to verify they're present
echo "Files in current directory:"
ls -lh {PRIORITY}_chr*_MAF{maf_code}.pgen

# Count chromosomes
chr_count=$(ls {PRIORITY}_chr*_MAF{maf_code}.pgen 2>/dev/null | wc -l)
echo ""
echo "Found $chr_count chromosome files"

if [ "$chr_count" -ne 22 ]; then
    echo "WARNING: Expected 22 chromosomes, found $chr_count"
fi

echo ""
echo "Merging with PLINK2..."
plink2 --pfile {PRIORITY}_chr1_MAF{maf_code} \\
       --pmerge-list merge_{PRIORITY}_MAF{maf_code}.txt \\
       --make-pgen \\
       --out {PRIORITY}_merged_MAF{maf_code}

echo ""
echo " Merge complete!"
echo "Total variants: $(tail -n+2 {PRIORITY}_merged_MAF{maf_code}.pvar | wc -l)"
echo "Total samples: $(tail -n+2 {PRIORITY}_merged_MAF{maf_code}.psam | wc -l)"
"""
    
    script_name = f"merge_{PRIORITY}_MAF{maf_code}.sh"
    with open(script_name, 'w') as f:
        f.write(merge_script)
    
    # Build -iin arguments for all chromosome files (1-22)
    iin_args = []
    for chr_num in range(1, 23):
        iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}_QC/{PRIORITY}_chr{chr_num}_MAF{maf_code}.pgen"')
        iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}_QC/{PRIORITY}_chr{chr_num}_MAF{maf_code}.pvar"')
        iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}_QC/{PRIORITY}_chr{chr_num}_MAF{maf_code}.psam"')
    
    # Add merge list file
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merge_{PRIORITY}_MAF{maf_code}.txt"')
    
    # Join all iin arguments
    iin_str = " ".join(iin_args)
    
    # Build dependency string
    depends_str = " ".join([f"--depends-on {j}" for j in qc_jobs])
    
    # Submit merge job with all input files
    job_cmd = f"""dx run swiss-army-knife {iin_str} -icmd="$(cat {script_name})" {depends_str} --instance-type mem2_ssd1_v2_x96 --destination /Results/burden_geno/{PRIORITY}/merged/ --name merge_{PRIORITY}_MAF{maf_code} --yes --brief"""
    
    print(f"  Submitting job with {len(iin_args)} input files...")
    result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"   Error submitting job for MAF{maf_code}:")
        print(f"  stderr: {result.stderr}")
        print(f"  stdout: {result.stdout}")
        continue
    
    job_id = result.stdout.strip()
    
    if not job_id or not job_id.startswith("job-"):
        print(f"   Invalid job ID returned: {job_id}")
        print(f"  stderr: {result.stderr}")
        continue
    
    merge_jobs[maf_code] = job_id
    print(f"   Job submitted: {job_id}")

if len(merge_jobs) == 3:
    print(f"\n Successfully submitted all 3 merge jobs")
else:
    print(f"\n Only submitted {len(merge_jobs)}/3 merge jobs")
    print(f"  Successful: {list(merge_jobs.keys())}")

print(" These will start automatically after all QC jobs complete")
print(f"\nMonitor progress: dx watch {list(merge_jobs.values())[0]}" if merge_jobs else "\nNo jobs to monitor")


## Extracting coding variants for {PRIORITY}

In [ ]:
# Check if merge_jobs exists
if 'merge_jobs' not in locals() and 'merge_jobs' not in globals():
    print("\n merge_jobs not found. Attempting to find merge job IDs...")
    merge_jobs = {}
    
    for maf_code in MAF_THRESHOLDS.keys():
        job_name = f"merge_{PRIORITY}_MAF{maf_code}"
        result = subprocess.run(
            f"dx find jobs --name '{job_name}' --created-after=-7d --brief --state done",
            shell=True,
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0 and result.stdout.strip():
            job_ids = result.stdout.strip().split('\n')
            if job_ids and job_ids[0]:
                merge_jobs[maf_code] = job_ids[0]
                print(f"  Found MAF{maf_code}: {job_ids[0]}")
        else:
            print(f"   Could not find completed merge job for MAF{maf_code}")
    
    if not merge_jobs:
        print("\n ERROR: No merge jobs found!")
        raise RuntimeError("merge_jobs not available")
    
    print(f"\n Found {len(merge_jobs)}/3 merge jobs")

coding_script = f"""#!/bin/bash
set -euo pipefail

# Download GENCODE annotations
echo "Downloading GENCODE v38 annotations..."
wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz

# Extract CDS regions to BED format
echo "Extracting CDS regions..."
zcat gencode.v38.annotation.gtf.gz | \\
  awk '$3=="CDS"' | \\
  awk -v OFS='\\t' '{{print $1, $4-1, $5}}' | \\
  sort -k1,1 -k2,2n | \\
  bedtools merge -i - > cds_hg38.bed

echo " CDS BED file created: $(wc -l < cds_hg38.bed) regions"
echo ""

# List available files
echo "Available merged files:"
ls -lh {PRIORITY}_merged_MAF*.pgen

# Process each MAF threshold
for maf in 001 003 005; do
    echo ""
    echo "========================================"
    echo "Processing MAF ${{maf}}"
    echo "========================================"
    
    # Extract coding variants
    echo "Extracting coding variants..."
    plink2 --pfile {PRIORITY}_merged_MAF${{maf}} \\
           --extract bed1 cds_hg38.bed \\
           --make-pgen \\
           --out {PRIORITY}_merged_MAF${{maf}}_coding
    
    # Report statistics
    total=$(tail -n+2 {PRIORITY}_merged_MAF${{maf}}.pvar | wc -l)
    coding=$(tail -n+2 {PRIORITY}_merged_MAF${{maf}}_coding.pvar | wc -l)
    pct=$(echo "scale=1; $coding*100/$total" | bc)
    
    echo ""
    echo " MAF ${{maf}}: ${{coding}}/${{total}} variants are coding (${{pct}}%)"
done

echo ""
echo "========================================"
echo " All coding extraction complete!"
echo "========================================"
"""

with open(f"extract_coding_{PRIORITY}.sh", 'w') as f:
    f.write(coding_script)

# Build -iin arguments for all 3 merged MAF files
iin_args = []
for maf_code in MAF_THRESHOLDS.keys():
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}.pgen"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}.pvar"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}.psam"')

iin_str = " ".join(iin_args)

# Submit coding extraction job 
depends_str = " ".join([f"--depends-on {j}" for j in merge_jobs.values()])

coding_cmd = f"""dx run swiss-army-knife {iin_str} -icmd="$(cat extract_coding_{PRIORITY}.sh)" {depends_str} --instance-type mem2_ssd1_v2_x96 --destination /Results/burden_geno/{PRIORITY}/merged/ --name extract_coding_{PRIORITY} --yes --brief"""

print(f"\nSubmitting coding extraction job with {len(iin_args)} input files...")
result = subprocess.run(coding_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print(f" Error submitting job:")
    print(f"  stderr: {result.stderr}")
    print(f"  stdout: {result.stdout}")
    coding_job_id = None
else:
    coding_job_id = result.stdout.strip()
    if coding_job_id and coding_job_id.startswith("job-"):
        print(f" Coding extraction job submitted: {coding_job_id}")
        print(" This will start automatically after all merges complete")
    else:
        print(f" Invalid job ID returned: {coding_job_id}")
        print(f"  stderr: {result.stderr}")
        coding_job_id = None

## Creating gene annotation files for REGENIE

In [ ]:
%%bash
dx mkdir -p /Results/burden_anno/EUR_bedtools/

In [ ]:
annotation_script = f"""#!/bin/bash
set -euo pipefail

echo "Creating gene annotation files for REGENIE burden testing..."

# Install required Python packages
echo "Installing required Python packages..."
pip install pandas --break-system-packages -q

# Download GENCODE gene definitions
echo "Downloading GENCODE v38 annotations..."
wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz

# Extract protein-coding genes with boundaries
echo "Extracting protein-coding genes..."
zcat gencode.v38.annotation.gtf.gz | \\
  awk '$3=="gene"' | \\
  awk -v OFS='\\t' '{{
    match($0, /gene_name "([^"]+)"/, gname)
    match($0, /gene_type "([^"]+)"/, gtype)
    if(gtype[1]=="protein_coding") {{
      chr=$1
      gsub("chr", "", chr)
      print gname[1], chr, $4, $5
    }}
  }}' | sort -k1,1 > genes_hg38.txt

echo " Extracted $(wc -l < genes_hg38.txt) protein-coding genes"

# Process each MAF × variant type combination
for maf in 001 003 005; do
    for vtype in all coding; do
        echo ""
        echo "=========================================="
        echo "Processing MAF ${{maf}} - ${{vtype}} variants"
        echo "=========================================="
        
        # Determine file prefix
        if [ "$vtype" == "all" ]; then
            pfx="{PRIORITY}_merged_MAF${{maf}}"
        else
            pfx="{PRIORITY}_merged_MAF${{maf}}_coding"
        fi
        
        echo "Using genotype files: ${{pfx}}.pgen/pvar/psam"
        
        # Convert PVAR to BED format (chr, start, end, id, ref, alt)
        echo "Converting PVAR to BED format..."
        awk 'NR>1 {{print $1 "\\t" ($2-1) "\\t" $2 "\\t" $3 "\\t" $4 "\\t" $5}}' ${{pfx}}.pvar > ${{pfx}}_variants.bed
        
        # Convert genes to BED format (chr, start, end, gene_name)
        echo "Converting genes to BED format..."
        awk '{{print $2 "\\t" $3 "\\t" $4 "\\t" $1}}' genes_hg38.txt > genes_hg38.bed
        
        # Use bedtools intersect to map variants to genes 
        echo "Mapping variants to genes with bedtools..."
        bedtools intersect -a ${{pfx}}_variants.bed -b genes_hg38.bed -wa -wb | \\
          awk '{{print $1 "\\t" ($2+1) "\\t" $5 "\\t" $6 "\\t" $10}}' > ${{pfx}}_anno.txt
        
        # Count variants and genes
        var_count=$(wc -l < ${{pfx}}_anno.txt)
        gene_count=$(cut -f5 ${{pfx}}_anno.txt | sort -u | wc -l)
        
        echo "   Created ${{var_count}} variant-gene annotations"
        echo "   Genes with variants: ${{gene_count}}"
        
        # Create set list file (one gene per line)
        echo "Creating set list..."
        cut -f5 ${{pfx}}_anno.txt | sort -u > ${{pfx}}_sets.txt
        echo "   Gene sets: $(wc -l < ${{pfx}}_sets.txt)"
        
        # Create mask definition file
        echo "Creating mask file..."
        cat > ${{pfx}}_mask.txt << 'MASKEOF'
# REGENIE mask definition file
# Format: MASK_NAME CONSEQUENCE_TYPES
LOF stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant
MISSENSE missense_variant
DAMAGING stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant,missense_variant
MASKEOF
        
        echo "   Mask file created"
        
        # Clean up temporary BED files
        rm -f ${{pfx}}_variants.bed
        
        echo ""
        echo "Summary for MAF ${{maf}} - ${{vtype}}:"
        echo "  Annotation file: ${{pfx}}_anno.txt"
        echo "  Set list: ${{pfx}}_sets.txt"
        echo "  Mask file: ${{pfx}}_mask.txt"
    done
done

# Clean up
rm -f genes_hg38.bed

echo ""
echo " All annotation files created!"
"""

with open(f"create_annotations_{PRIORITY}.sh", 'w') as f:
    f.write(annotation_script)

# Check if merge_jobs and coding_job_id exist
if 'merge_jobs' not in locals() and 'merge_jobs' not in globals():
    print("\n merge_jobs not found. Finding from DNAnexus...")
    merge_jobs = {}
    for maf_code in MAF_THRESHOLDS.keys():
        result = subprocess.run(
            f"dx find jobs --name 'merge_{PRIORITY}_MAF{maf_code}' --created-after=-7d --brief --state done",
            shell=True, capture_output=True, text=True
        )
        if result.returncode == 0 and result.stdout.strip():
            merge_jobs[maf_code] = result.stdout.strip().split('\n')[0]

if 'coding_job_id' not in locals() and 'coding_job_id' not in globals():
    print(" coding_job_id not found. Finding from DNAnexus...")
    result = subprocess.run(
        f"dx find jobs --name 'extract_coding_{PRIORITY}' --created-after=-7d --brief --state done",
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0 and result.stdout.strip():
        coding_job_id = result.stdout.strip().split('\n')[0]
        print(f"  Found: {coding_job_id}")
    else:
        print("   Could not find coding extraction job")
        print("  Run Part 11 first or set manually: coding_job_id = 'job-xxx'")
        coding_job_id = None

if not merge_jobs or not coding_job_id:
    print("\n ERROR: Required job IDs not found!")
    raise RuntimeError("Missing required job IDs")

# Build -iin arguments for all 6 combinations (3 MAF × 2 types)
iin_args = []
for maf_code in MAF_THRESHOLDS.keys():
    # All variants
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}.pgen"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}.pvar"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}.psam"')
    # Coding variants
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}_coding.pgen"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}_coding.pvar"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/merged/{PRIORITY}_merged_MAF{maf_code}_coding.psam"')

iin_str = " ".join(iin_args)

# Submit annotation job 
depends_str = " ".join([f"--depends-on {j}" for j in list(merge_jobs.values()) + [coding_job_id]])

# Use unique job name with timestamp to allow parallel runs
import time
timestamp = int(time.time())
job_name = f"annotate_{PRIORITY}_bedtools_{timestamp}"

# Save to separate folder for comparison
output_dir = f"/Results/burden_anno/{PRIORITY}_bedtools/"

anno_cmd = f"""dx run swiss-army-knife {iin_str} -icmd="$(cat create_annotations_{PRIORITY}.sh)" {depends_str} --instance-type mem2_ssd1_v2_x96 --destination {output_dir} --name {job_name} --yes --brief"""

print(f"\nSubmitting annotation job with {len(iin_args)} input files...")
print(f"Job name: {job_name}")

result = subprocess.run(anno_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print(f" Error submitting job:")
    print(f"  stderr: {result.stderr}")
    print(f"  stdout: {result.stdout}")
    anno_job_id = None
else:
    anno_job_id = result.stdout.strip()
    if anno_job_id and anno_job_id.startswith("job-"):
        print(f"\n Annotation job submitted: {anno_job_id}")
        print(" This will start automatically after merges and coding extraction complete")
    else:
        print(f" Invalid job ID returned: {anno_job_id}")
        print(f"  stderr: {result.stderr}")
        anno_job_id = None

## Extract Common Variants for REGENIE Step 1

In [ ]:
%%bash
dx mkdir -p /Results/burden_geno/EUR/step1_input

In [ ]:
#!/usr/bin/env python3
import subprocess

PRIORITY = "EUR"  

# Create script to extract and prune common variants
common_script = f"""#!/bin/bash
set -euo pipefail

# Process each chromosome
for chr in {{1..22}}; do
    echo "Processing chr${{chr}}..."
    
    # Extract common variants from this chromosome
    plink2 --pfile {PRIORITY}_chr${{chr}}_raw \\
           --maf 0.05 \\
           --mac 20 \\
           --geno 0.05 \\
           --hwe 1e-6 \\
           --make-pgen \\
           --out {PRIORITY}_chr${{chr}}_common
    
    var_count=$(tail -n+2 {PRIORITY}_chr${{chr}}_common.pvar | wc -l)
    echo "   Chr${{chr}}: ${{var_count}} common variants"
    echo ""
done


# Create merge list
for chr in {{1..22}}; do
    echo "{PRIORITY}_chr${{chr}}_common" >> merge_common.txt
done

# Merge all chromosomes
plink2 --pmerge-list merge_common.txt \\
       --make-pgen \\
       --out {PRIORITY}_merged_common

common_count=$(tail -n+2 {PRIORITY}_merged_common.pvar | wc -l)
echo ""
echo " Total common variants merged: ${{common_count}}"
echo ""

# LD prune
plink2 --pfile {PRIORITY}_merged_common \\
       --indep-pairwise 1000 100 0.5 \\
       --out {PRIORITY}_common_pruned

if [ ! -f {PRIORITY}_common_pruned.prune.in ]; then
    echo " ERROR: LD pruning failed!"
    exit 1
fi

pruned_list_count=$(wc -l < {PRIORITY}_common_pruned.prune.in)
echo " Variants to keep after pruning: ${{pruned_list_count}}"
echo ""

# Extract pruned variants
plink2 --pfile {PRIORITY}_merged_common \\
       --extract {PRIORITY}_common_pruned.prune.in \\
       --make-pgen \\
       --out {PRIORITY}_step1_variants

pruned_count=$(tail -n+2 {PRIORITY}_step1_variants.pvar | wc -l)
echo ""
echo "Final LD-pruned variants for Step 1: ${{pruned_count}}"
echo ""
echo "Files created:"
echo "  {PRIORITY}_step1_variants.pgen"
echo "  {PRIORITY}_step1_variants.pvar"
echo "  {PRIORITY}_step1_variants.psam"
echo ""
echo "These are ready for REGENIE Step 1!"
"""

with open(f"extract_common_{PRIORITY}.sh", 'w') as f:
    f.write(common_script)

print(f" Script created: extract_common_{PRIORITY}.sh")
print("")

# Build -iin arguments for all chromosome raw files
print("Building input file list...")
iin_args = []
for chr_num in range(1, 23):
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}/{PRIORITY}_chr{chr_num}_raw.pgen"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}/{PRIORITY}_chr{chr_num}_raw.pvar"')
    iin_args.append(f'-iin="/Results/burden_geno/{PRIORITY}/chr{chr_num}/{PRIORITY}_chr{chr_num}_raw.psam"')

iin_str = " ".join(iin_args)

print(f" Using {len(iin_args)} input files (22 chromosomes × 3 files)")
print("")

# Submit job
common_cmd = f"""dx run swiss-army-knife {iin_str} -icmd="$(cat extract_common_{PRIORITY}.sh)" --instance-type mem2_ssd1_v2_x96 --destination /Results/burden_geno/{PRIORITY}/step1_input/ --name extract_common_{PRIORITY} --yes --brief"""

print("Submitting job to extract common variants...")
result = subprocess.run(common_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print(f" Error submitting job:")
    print(f"  stderr: {result.stderr}")
    print(f"  stdout: {result.stdout}")
else:
    job_id = result.stdout.strip()
    if job_id and job_id.startswith("job-"):
        print(f" Job submitted: {job_id}")
        print(f"  {PRIORITY}_step1_variants.pgen/pvar/psam")
    else:
        print(f" Invalid job ID returned: {job_id}")
        print(f"  stderr: {result.stderr}")


## Recodes phenotypes from 1/2 to 0/1

In [ ]:
#!/usr/bin/env python3

import subprocess
import pandas as pd

PRIORITY = "EUR"
OLD_FILE = f"PHENO_{PRIORITY}.txt"
NEW_FILE = f"PHENO_{PRIORITY}_01coded.txt"

print("Step 1: Downloading existing phenotype file...")
result = subprocess.run(
    f'dx download "/Results/burden_pheno/{OLD_FILE}"',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f" Error downloading file:")
    print(f"  {result.stderr}")
    exit(1)

print(f" Downloaded {OLD_FILE}")

print("Step 2: Reading phenotype file...")
df = pd.read_csv(OLD_FILE, sep="\t")

print(f" Loaded {len(df)} participants")
print("Current phenotype distribution:")
print(df['PHENO'].value_counts().sort_index())

print("Step 3: Recoding phenotypes...")

# Check what coding we have
pheno_min = df['PHENO'].min()
pheno_max = df['PHENO'].max()

if pheno_min == 1 and pheno_max == 2:
    print("  Detected 1/2 coding → Recoding to 0/1")
    df['PHENO'] = df['PHENO'] - 1  # 1→0, 2→1
    
elif pheno_min == 0 and pheno_max == 1:
    print("  Already 0/1 coded → No changes needed")
    
else:
    print(f"  Unexpected coding: min={pheno_min}, max={pheno_max}")
    print("  Proceeding anyway...")

print("New phenotype distribution:")
print(df['PHENO'].value_counts().sort_index())


print(f"Step 4: Saving as {NEW_FILE}...")
df.to_csv(NEW_FILE, sep="\t", index=False)
print(f" Saved locally")

print(f"Step 5: Uploading to DNAnexus...")

# Upload new file
result = subprocess.run(
    f'dx upload {NEW_FILE} --path /Results/burden_pheno/',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f" Error uploading:")
    print(f"  {result.stderr}")
    exit(1)

print(f" Uploaded to /Results/burden_pheno/{NEW_FILE}")

print("Step 6: Verifying upload...")
result = subprocess.run(
    f'dx ls /Results/burden_pheno/{NEW_FILE}',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(" File verified on DNAnexus")
else:
    print("  Could not verify (but upload likely succeeded)")

print(f"Participants: {len(df)}")
print(f"  Controls (PHENO=0): {(df['PHENO']==0).sum()}")
print(f"  Cases (PHENO=1): {(df['PHENO']==1).sum()}")
if df['PHENO'].isna().sum() > 0:
    print(f"  Missing (PHENO=NA): {df['PHENO'].isna().sum()}")


## REGENIE Step 1 - Fit Null Model

In [ ]:
%%bash
dx mkdir -p /Results/burden_results/EUR/step1

In [ ]:
#!/usr/bin/env python3

import subprocess

PRIORITY = "EUR"  
PHENO_FILE = f"PHENO_{PRIORITY}_01coded.txt"  

# Create Step 1 script
step1_script = f"""#!/bin/bash
set -euo pipefail


# Install REGENIE
echo "Installing REGENIE v3.2.9..."
wget -q https://github.com/rgcgithub/regenie/releases/download/v3.2.9/regenie_v3.2.9.gz_x86_64_Linux.zip
unzip -q regenie_v3.2.9.gz_x86_64_Linux.zip
mv regenie_v3.2.9.gz_x86_64_Linux regenie
chmod +x regenie

echo " REGENIE installed"
echo ""

# Files are already available via -iin 
echo "Input files available:"
ls -lh {PRIORITY}_step1_variants.pgen
ls -lh {PRIORITY}_step1_variants.pvar
ls -lh {PRIORITY}_step1_variants.psam
ls -lh {PHENO_FILE}
ls -lh covariates.tsv
echo ""

# Check variant count
var_count=$(tail -n+2 {PRIORITY}_step1_variants.pvar | wc -l)
echo "Using ${{var_count}} LD-pruned common variants"
echo ""

if [ "$var_count" -lt 1000 ]; then
    echo " ERROR: Too few variants (${{var_count}})"
    echo "Need at least 1000 variants for REGENIE Step 1"
    exit 1
fi

if [ "$var_count" -gt 1000000 ]; then
    echo "  WARNING: Too many variants (${{var_count}})"
    echo "REGENIE recommends <1M variants for Step 1"
    echo "Using --force-step1 to proceed..."
fi

# Run REGENIE Step 1

./regenie \\
  --step 1 \\
  --pgen {PRIORITY}_step1_variants \\
  --phenoFile {PHENO_FILE} \\
  --covarFile covariates.tsv \\
  --phenoCol PHENO \\
  --covarColList SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \\
  --bt \\
  --bsize 1000 \\
  --lowmem \\
  --lowmem-prefix tmp_{PRIORITY} \\
  --force-step1 \\
  --gz \\
  --threads 96 \\
  --out {PRIORITY}_step1

echo ""
echo " REGENIE Step 1 complete!"
echo ""

# Check LOCO files created
echo "Verifying LOCO files..."
loco_count=$(ls {PRIORITY}_step1_*.loco.gz 2>/dev/null | wc -l)
echo "LOCO files created: ${{loco_count}}/22"

if [ "$loco_count" -eq 22 ]; then
    echo " All 22 LOCO files created successfully!"
elif [ "$loco_count" -gt 0 ]; then
    echo "  WARNING: Only ${{loco_count}}/22 LOCO files created"
    echo "Missing LOCO files may cause Step 2 to fail"
    echo "You may need to re-run Step 1"
else
    echo " ERROR: No LOCO files created!"
    exit 1
fi

echo ""
echo "Output files:"
ls -lh {PRIORITY}_step1*

echo ""
echo "=== Summary from log ==="
grep "Number of individuals" {PRIORITY}_step1.log || true
grep "Number of analyzed phenotypes" {PRIORITY}_step1.log || true
grep "Number of genotyped SNPs" {PRIORITY}_step1.log || true
grep "Elapsed time" {PRIORITY}_step1.log || true
echo "========================"
"""

with open(f"step1_{PRIORITY}.sh", 'w') as f:
    f.write(step1_script)

print(" Step 1 script created")

# Build -iin arguments for Step 1
iin_args = [
    f'-iin="/Results/burden_geno/{PRIORITY}/step1_input/{PRIORITY}_step1_variants.pgen"',
    f'-iin="/Results/burden_geno/{PRIORITY}/step1_input/{PRIORITY}_step1_variants.pvar"',
    f'-iin="/Results/burden_geno/{PRIORITY}/step1_input/{PRIORITY}_step1_variants.psam"',
    f'-iin="/Results/burden_pheno/{PHENO_FILE}"',
    f'-iin="/Results/burden_pheno/covariates.tsv"'
]
iin_str = " ".join(iin_args)


# Submit Step 1 job
step1_cmd = f"""dx run swiss-army-knife {iin_str} -icmd="$(cat step1_{PRIORITY}.sh)" --instance-type mem2_ssd1_v2_x96 --destination /Results/burden_results/{PRIORITY}/step1/ --name step1_{PRIORITY}_fixed --yes --brief"""

print(f"Submitting Step 1 job with {len(iin_args)} input files...")

result = subprocess.run(step1_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print(f" Error submitting Step 1 job:")
    print(f"  stderr: {result.stderr}")
    print(f"  stdout: {result.stdout}")
    print("")
    if "does not exist" in result.stderr or "not found" in result.stderr:
        print("  This error likely means previous Part hasn't completed yet.")
    step1_job_id = None
else:
    step1_job_id = result.stdout.strip()
    if step1_job_id and step1_job_id.startswith("job-"):
        print(f" Step 1 job submitted: {step1_job_id}")
    else:
        print(f" Invalid job ID returned: {step1_job_id}")
        print(f"  stderr: {result.stderr}")
        step1_job_id = None


## Fix Bedtools Annotations for REGENIE

In [ ]:
#!/usr/bin/env python3

import subprocess
import gzip

PRIORITY = "EUR"

MAF_CODES = ['001', '003', '005']
VTYPES = ['', '_coding']


for maf_code in MAF_CODES:
    for vtype_suffix in VTYPES:
        
        vtype_name = "all" if vtype_suffix == "" else "coding"
        pfx = f"{PRIORITY}_merged_MAF{maf_code}{vtype_suffix}"
        
        print(f"Processing: {pfx}")
        
        # Create fixing script
        fix_script = f"""#!/bin/bash
set -euo pipefail

echo "Fixing annotations for {pfx}..."
echo "Files available via -iin:"

ls -lh {pfx}_anno.txt
ls -lh {pfx}_sets.txt
ls -lh {pfx}_mask.txt

echo ""
echo " Input files available"

# The bedtools annotation format is: CHR POS REF ALT GENE
# We need to convert to: DRAGEN:chrCHR:POS:REF:ALT GENE
# To match the .pvar ID format

echo "Converting annotation format to match DRAGEN IDs..."

awk 'BEGIN{{OFS="\\t"}} {{print "DRAGEN:chr"$1":"$2":"$3":"$4, $5}}' {pfx}_anno.txt > {pfx}_anno_fixed.txt

echo " Converted $(wc -l < {pfx}_anno_fixed.txt) variants"

# Sets and mask files don't need changes (no variant IDs)
cp {pfx}_sets.txt {pfx}_sets_fixed.txt
cp {pfx}_mask.txt {pfx}_mask_fixed.txt

# Show sample
echo ""
echo "Sample of fixed annotation (first 5 lines):"
head -5 {pfx}_anno_fixed.txt
echo ""
echo "Format: DRAGEN:chr1:10417:C:G GENE"
echo "This matches .pvar ID column!"
"""
        
        script_name = f"fix_anno_{pfx}.sh"
        with open(script_name, 'w') as f:
            f.write(fix_script)
        
        # Build -iin for the 3 bedtools files
        iin_args = [
            f'-iin="/Results/burden_anno/{PRIORITY}_bedtools/{pfx}_anno.txt"',
            f'-iin="/Results/burden_anno/{PRIORITY}_bedtools/{pfx}_sets.txt"',
            f'-iin="/Results/burden_anno/{PRIORITY}_bedtools/{pfx}_mask.txt"'
        ]
        iin_str = " ".join(iin_args)
        
        # Submit job with -iin
        job_cmd = f'dx run swiss-army-knife {iin_str} -icmd="$(cat {script_name})" --instance-type mem1_ssd1_v2_x2 --destination /Results/burden_anno/{PRIORITY}_bedtools_fixed/ --name fix_anno_{pfx} --yes --brief'
        
        result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            job_id = result.stdout.strip()
            print(f"   Job submitted: {job_id}")
        else:
            print(f"  Error: {result.stderr}")

## Fix Annotation File Order for REGENIE

In [ ]:
#!/usr/bin/env python3

import subprocess

PRIORITY = "EUR"

MAF_CODES = ['001', '003', '005']
VTYPES = ['', '_coding']


for maf_code in MAF_CODES:
    for vtype_suffix in VTYPES:
        
        vtype_name = "all" if vtype_suffix == "" else "coding"
        pfx = f"{PRIORITY}_merged_MAF{maf_code}{vtype_suffix}"
        
        print(f"Processing: {pfx}")
        
        # Create fixing script
        fix_script = f"""#!/bin/bash
set -euo pipefail

echo "Fixing annotation format for {pfx}..."
echo "Files available via -iin:"

ls -lh {pfx}_anno_fixed.txt
ls -lh {pfx}_sets_fixed.txt
ls -lh {pfx}_mask_fixed.txt

echo ""
echo "Current format: VARIANT<TAB>GENE (2 columns)"
echo "REGENIE needs:  VARIANT<TAB>ANNOTATION<TAB>GENE (3 columns)"
echo ""
echo "Adding '.' as annotation type (includes all variants)..."

# Add "." as the annotation column (column 2)
# Input:  VARIANT GENE
# Output: VARIANT . GENE
awk 'BEGIN{{OFS="\\t"}} {{print $1, ".", $2}}' {pfx}_anno_fixed.txt > {pfx}_anno_regenie.txt

echo " Converted $(wc -l < {pfx}_anno_regenie.txt) variants"

# Update mask file to use "." annotation
echo "M1	." > {pfx}_mask_regenie.txt
echo " Updated mask file to use '.' (all variants)"

# Sets file doesn't change
cp {pfx}_sets_fixed.txt {pfx}_sets_regenie.txt

# Show samples
echo ""
echo "Sample of fixed annotation (first 5 lines):"
head -5 {pfx}_anno_regenie.txt
echo ""
echo "Mask file content:"
cat {pfx}_mask_regenie.txt
echo ""
echo "Format: VARIANT<TAB>.<TAB>GENE (3 columns, correct for REGENIE!)"
"""
        
        script_name = f"fix_anno3col_{pfx}.sh"
        with open(script_name, 'w') as f:
            f.write(fix_script)
        
        # Build -iin for the 3 _fixed files
        iin_args = [
            f'-iin="/Results/burden_anno/EUR_bedtools_fixed/{pfx}_anno_fixed.txt"',
            f'-iin="/Results/burden_anno/EUR_bedtools_fixed/{pfx}_sets_fixed.txt"',
            f'-iin="/Results/burden_anno/EUR_bedtools_fixed/{pfx}_mask_fixed.txt"'
        ]
        iin_str = " ".join(iin_args)
        
        # Submit job
        job_cmd = f'dx run swiss-army-knife {iin_str} -icmd="$(cat {script_name})" --instance-type mem1_ssd1_v2_x2 --destination /Results/burden_anno/{PRIORITY}_regenie/ --name fix_anno3col_{pfx} --yes --brief'
        
        result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            job_id = result.stdout.strip()
            print(f"   Job submitted: {job_id}")
        else:
            print(f"   Error: {result.stderr}")

## Fix Annotation Files - Replace . with M1

In [ ]:
#!/usr/bin/env python3

import subprocess

PRIORITY = "EUR"

MAF_CODES = ['001', '003', '005']
VTYPES = ['', '_coding']


# Create the output folder first
print("Creating output folder...")
result = subprocess.run(
    'dx mkdir -p /Results/burden_anno/EUR_M1/',
    shell=True,
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print(" Folder created: /Results/burden_anno/EUR_M1/")
else:
    print(f"Note: {result.stderr}")


for maf_code in MAF_CODES:
    for vtype_suffix in VTYPES:
        
        vtype_name = "all" if vtype_suffix == "" else "coding"
        pfx = f"{PRIORITY}_merged_MAF{maf_code}{vtype_suffix}"
        
        print(f"Processing: {pfx}")
        
        # Create fixing script
        fix_script = f"""#!/bin/bash
set -euo pipefail

echo "Fixing annotation for {pfx}..."
echo "Files available:"
ls -lh

echo ""
echo "Changing: VARIANT . GENE"
echo "To:       VARIANT M1 GENE"

# Replace "." with "M1" in column 2
awk 'BEGIN{{OFS="\\t"}} {{$2="M1"; print}}' {pfx}_anno_regenie.txt > {pfx}_anno_fixed.txt

echo " Fixed $(wc -l < {pfx}_anno_fixed.txt) variants"

# Keep sets and mask files as-is
cp {pfx}_sets_regenie.txt {pfx}_sets_fixed.txt
cp {pfx}_mask_regenie.txt {pfx}_mask_fixed.txt

echo ""
echo "Output files created:"
ls -lh {pfx}_anno_fixed.txt
ls -lh {pfx}_sets_fixed.txt
ls -lh {pfx}_mask_fixed.txt

echo ""
echo "Sample of fixed annotation (first 5 lines):"
head -5 {pfx}_anno_fixed.txt

echo ""
echo "Format: VARIANT<TAB>M1<TAB>GENE"
"""
        
        script_name = f"fix_m1_{pfx}.sh"
        with open(script_name, 'w') as f:
            f.write(fix_script)
        
        # Build -iin for the 3 files
        iin_args = [
            f'-iin="/Results/burden_anno/{PRIORITY}_regenie/{pfx}_anno_regenie.txt"',
            f'-iin="/Results/burden_anno/{PRIORITY}_regenie/{pfx}_sets_regenie.txt"',
            f'-iin="/Results/burden_anno/{PRIORITY}_regenie/{pfx}_mask_regenie.txt"'
        ]
        iin_str = " ".join(iin_args)
        
        # Submit job
        job_cmd = f'dx run swiss-army-knife {iin_str} -icmd="$(cat {script_name})" --instance-type mem1_ssd1_v2_x2 --destination /Results/burden_anno/{PRIORITY}_M1/ --name fix_m1_{pfx} --yes --brief'
        
        result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            job_id = result.stdout.strip()
            print(f"   Job submitted: {job_id}")
        else:
            print(f"   Error: {result.stderr}")

## Fix Annotations to Proper 4-Column REGENIE Format

In [ ]:
#!/usr/bin/env python3

import subprocess

PRIORITY = "EUR"

MAF_CODES = ['001', '003', '005']
VTYPES = ['', '_coding']


# Create output folder
print("Creating output folder...")
subprocess.run('dx mkdir -p /Results/burden_anno/EUR_proper/', shell=True)
print(" Folder created")

for maf_code in MAF_CODES:
    for vtype_suffix in VTYPES:
        
        vtype_name = "all" if vtype_suffix == "" else "coding"
        pfx = f"{PRIORITY}_merged_MAF{maf_code}{vtype_suffix}"
        
        print(f"Processing: {pfx}")
        
        # Create fixing script
        fix_script = f"""#!/bin/bash
set -euo pipefail

echo "Converting to proper 4-column format for {pfx}..."
echo ""

# Current format: VARIANT M1 GENE (3 columns)
# Need format:    VARIANT GENE REGION ANNOTATION (4 columns)

# Since we don't have region info, use "." for region
# Since we don't have specific annotations, use "ALL" for annotation type

awk 'BEGIN{{OFS="\\t"}}
!seen[$1]++ {{
    variant = $1
    gene = $3
    region = "."
    annotation = "ALL"
    print variant, gene, region, annotation
}}' {pfx}_anno_fixed.txt > {pfx}_anno_proper.txt

echo " Created 4-column annotation: $(wc -l < {pfx}_anno_proper.txt) variants"

# Create set-list with coordinates
# Format: GENE CHR START VARIANT1,VARIANT2,...

awk 'BEGIN{{OFS="\\t"}}
!seen[$1]++ {{
    variant = $1
    gene = $3
    
    # Extract chr and pos from DRAGEN:chr1:12345:A:T
    split(variant, parts, ":")
    if (parts[1] == "DRAGEN") {{
        chr = parts[2]
        gsub("chr", "", chr)
        pos = parts[3]
    }} else {{
        chr = "1"
        pos = "0"
    }}
    
    # Build variant list per gene
    if (gene in genes) {{
        genes[gene] = genes[gene] "," variant
        if (pos < starts[gene]) starts[gene] = pos
    }} else {{
        genes[gene] = variant
        chrs[gene] = chr
        starts[gene] = pos
    }}
}}
END {{
    for (gene in genes) {{
        print gene, chrs[gene], starts[gene], genes[gene]
    }}
}}' {pfx}_anno_fixed.txt > {pfx}_sets_proper.txt

echo " Created set-list: $(wc -l < {pfx}_sets_proper.txt) genes"

# Create simple mask file
# Since we labeled everything as "ALL", mask should include "ALL"
echo -e "M1\\tALL" > {pfx}_mask_proper.txt

echo " Created mask file"

echo ""
echo "Sample annotation (first 3 lines):"
head -3 {pfx}_anno_proper.txt

echo ""
echo "Sample set-list (first 3 lines):"
head -3 {pfx}_sets_proper.txt

echo ""
echo "Mask file:"
cat {pfx}_mask_proper.txt

echo ""
echo "Format verified: 4 columns (VARIANT GENE REGION ANNOTATION)"
"""
        
        script_name = f"fix_proper_{pfx}.sh"
        with open(script_name, 'w') as f:
            f.write(fix_script)
        
        # Build -iin
        iin_args = [
            f'-iin="/Results/burden_anno/{PRIORITY}_M1/{pfx}_anno_fixed.txt"'
        ]
        iin_str = " ".join(iin_args)
        
        # Submit job
        job_cmd = f'dx run swiss-army-knife {iin_str} -icmd="$(cat {script_name})" --instance-type mem1_ssd1_v2_x2 --destination /Results/burden_anno/{PRIORITY}_proper/ --name fix_proper_{pfx} --yes --brief'
        
        result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            job_id = result.stdout.strip()
            print(f"   Job submitted: {job_id}")
        else:
            print(f"   Error: {result.stderr}")

## REGENIE Step 2

In [ ]:
%%bash
dx mkdir -p /Results/burden_results/EUR/MAF001/all
dx mkdir -p /Results/burden_results/EUR/MAF001/coding
dx mkdir -p /Results/burden_results/EUR/MAF003/all
dx mkdir -p /Results/burden_results/EUR/MAF003/coding
dx mkdir -p /Results/burden_results/EUR/MAF005/all
dx mkdir -p /Results/burden_results/EUR/MAF005/coding

In [ ]:
#!/usr/bin/env python3

import subprocess

PRIORITY = "EUR"
PHENO_FILE = f"PHENO_{PRIORITY}_01coded.txt"

MAF_THRESHOLDS = {
    '001': '0.01',
    '003': '0.03',
    '005': '0.05'
}

VARIANT_TYPES = ['all', 'coding']

step2_jobs = {}


for maf_code, maf_value in MAF_THRESHOLDS.items():
    for vtype in VARIANT_TYPES:

        print(f"\n--- Preparing Step 2: MAF {maf_value} - {vtype} ---")

        pgen_suffix = "" if vtype == "all" else "_coding"
        pfx = f"{PRIORITY}_merged_MAF{maf_code}{pgen_suffix}"

        step2_script = f"""#!/bin/bash
set -euxo pipefail


# Install REGENIE
if [ ! -f "./regenie" ]; then
    wget -q https://github.com/rgcgithub/regenie/releases/download/v3.2.9/regenie_v3.2.9.gz_x86_64_Linux.zip
    unzip -q regenie_v3.2.9.gz_x86_64_Linux.zip
    mv regenie_v3.2.9.gz_x86_64_Linux regenie
    chmod +x regenie
fi

echo "Files available:"
ls -lh

echo ""
echo "Checking annotation format (first 3 lines):"
head -3 {pfx}_anno_proper.txt
echo "Format: VARIANT GENE REGION ANNOTATION (4 columns)"

echo ""
echo "Checking set-list format (first 3 lines):"
head -3 {pfx}_sets_proper.txt
echo "Format: GENE CHR START VARIANT_LIST (4 columns)"

echo ""
echo "Mask file:"
cat {pfx}_mask_proper.txt
echo "Format: MASK_NAME ANNOTATION_TYPES"

echo ""
echo "Running REGENIE Step 2 with proper format..."
./regenie \\
  --step 2 \\
  --pgen "{pfx}" \\
  --phenoFile "{PHENO_FILE}" \\
  --covarFile "covariates.tsv" \\
  --phenoCol PHENO \\
  --covarColList SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \\
  --bt \\
  --pred "{PRIORITY}_step1_pred.list" \\
  --anno-file {pfx}_anno_proper.txt \\
  --set-list {pfx}_sets_proper.txt \\
  --mask-def {pfx}_mask_proper.txt \\
  --vc-tests skato \\
  --threads 96 \\
  --gz \\
  --out "{PRIORITY}_MAF{maf_code}_{vtype}_burden"

echo ""
echo " REGENIE Step 2 complete!"
echo ""
echo "Output files:"
ls -lh {PRIORITY}_MAF{maf_code}_{vtype}_burden*
"""

        # Save script
        script_name = f"step2_{PRIORITY}_MAF{maf_code}_{vtype}.sh"
        with open(script_name, 'w') as f:
            f.write(step2_script)

        # Build -iin arguments
        iin_args = [
            f'-iin="/Results/burden_geno/{PRIORITY}/merged/{pfx}.pgen"',
            f'-iin="/Results/burden_geno/{PRIORITY}/merged/{pfx}.pvar"',
            f'-iin="/Results/burden_geno/{PRIORITY}/merged/{pfx}.psam"',
            f'-iin="/Results/burden_anno/{PRIORITY}_proper/{pfx}_anno_proper.txt"',
            f'-iin="/Results/burden_anno/{PRIORITY}_proper/{pfx}_sets_proper.txt"',
            f'-iin="/Results/burden_anno/{PRIORITY}_proper/{pfx}_mask_proper.txt"',
            f'-iin="/Results/burden_pheno/{PHENO_FILE}"',
            f'-iin="/Results/burden_pheno/covariates.tsv"',
            f'-iin="/Results/burden_results/{PRIORITY}/step1/{PRIORITY}_step1_pred.list"',
            f'-iin="/Results/burden_results/{PRIORITY}/step1/{PRIORITY}_step1_1.loco.gz"'
        ]
        iin_str = " ".join(iin_args)

        # Submit job
        job_cmd = f"""dx run swiss-army-knife {iin_str} -icmd="$(cat {script_name})" --instance-type mem2_ssd1_v2_x96 --destination /Results/burden_results/{PRIORITY}/MAF{maf_code}/{vtype}/ --name step2_{PRIORITY}_MAF{maf_code}_{vtype} --yes --brief"""

        print(f"Submitting with {len(iin_args)} input files...")
        result = subprocess.run(job_cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            print(f" Error: {result.stderr}")
            job_id = None
        else:
            job_id = result.stdout.strip()
            if job_id and job_id.startswith("job-"):
                print(f" Job submitted: {job_id}")
                step2_jobs[f"MAF{maf_code}_{vtype}"] = job_id
            else:
                print(f" Invalid job ID: {job_id}")

print(f"\n Submitted {len(step2_jobs)}/6 jobs!")
for combo, job_id in step2_jobs.items():
    print(f"  {combo}: {job_id}")
